## Microsoft Security Incident Prediction - Qucik analisys

In [ ]:
# ── Librerie standard ──────────────────────────────────────────────────────
import time
import os
import warnings
warnings.filterwarnings("ignore") # sopprime i messaggi di deprecazione di Spark

# ── Manipolazione dati ─────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualizzazione ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# ── Machine Learning (Spark) ───────────────────────────────────────────────
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import (
    StringIndexer, VectorAssembler,
    ChiSqSelector, UnivariateFeatureSelector,
    VarianceThresholdSelector,
)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# ── Stile globale dei grafici ──────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 130, "font.family": "DejaVu Sans"})

OUTPUT_DIR = "images/Results/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

### Analisi

#### Step 2
**Perché usiamo Spark?**

Il dataset MetroPT-3 contiene ~15 milioni di righe raccolte a 1Hz per 6 mesi.
Elaborarlo in locale con Pandas rischierebbe crash o tempi molto lunghi.
Spark distribuisce il lavoro sui **4 worker del cluster**, ognuno con 4 core e 16 GB di RAM.

**Configurazione cluster**

| Parametro | Valore | Motivazione |
|---|---|---|
| `num_executors` | 4 | Un executor per worker |
| `executor_cores` | 4 | Tutti i core disponibili per worker |
| `executor_memory` | 13g | Lasciamo ~3g al SO per evitare OOM |
| `driver_memory` | 8g | Il driver raccoglie i risultati (feature importance, plot) |
| `shuffle.partitions` | 32 | 2 × core totali (16). Il default 200 è pensato per cluster molto più grandi |

**Costruzione del target**
Il dataset è **unlabeled by design**: non esiste una colonna target pronta.
Costruiamo la variabile `target` confrontando il timestamp di ogni riga con le
**4 finestre di guasto** fornite dall'azienda nel report tecnico:

| # | Inizio | Fine | Tipo |
|---|---|---|---|
| 1 | 18/04/2020 00:00 | 18/04/2020 23:59 | Air Leak |
| 2 | 29/05/2020 23:30 | 30/05/2020 06:00 | Air Leak |
| 3 | 05/06/2020 10:00 | 07/06/2020 14:30 | Air Leak |
| 4 | 15/07/2020 14:30 | 15/07/2020 19:00 | Air Leak |

- `target = 1` → guasto in corso
- `target = 0` → funzionamento normale

Il dataset risultante sarà **fortemente sbilanciato**: i guasti coprono poche ore
su 6 mesi di dati. Per questo motivo useremo **F1-score** come metrica principale
invece della semplice accuracy, che sarebbe fuorviante.

**Perché `.cache()`?**
Senza cache, ogni operazione sul DataFrame rileggerebbe il CSV da disco.
Con `.cache()` Spark mantiene il dataset in memoria dopo il primo accesso,
rendendo tutte le operazioni successive molto più veloci.

In [ ]:
# ── Configurazione cluster ─────────────────────────────────────────────────
MASTER_URL   = "spark://10.0.1.8:7077"
DRIVER_HOST  = "10.0.1.8"

spark = SparkSession.builder \
    .appName("MetroPT_Analysis") \
    .master(MASTER_URL) \
    .config("spark.submit.deployMode",      "client") \
    .config("spark.executor.instances",     "4") \
    .config("spark.executor.cores",         "4") \
    .config("spark.executor.memory",        "13g") \
    .config("spark.driver.memory",          "8g") \
    .config("spark.driver.host",            DRIVER_HOST) \
    .config("spark.driver.bindAddress",     DRIVER_HOST) \
    .config("spark.sql.shuffle.partitions", "32") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("SparkSession creata — versione:", spark.version)

# ── Caricamento dataset ────────────────────────────────────────────────────
TRAIN_PATH = "file:///home/PuckTrickadmin/DATASETS/MetroDT.csv"

t0 = time.time()
df_raw = spark.read.csv(TRAIN_PATH, header=True, inferSchema=True)

# ── Costruzione target dalle finestre di guasto ────────────────────────────
# Il dataset è unlabeled: costruiamo noi la colonna target confrontando
# il timestamp di ogni riga con gli intervalli di guasto noti dal report aziendale.
# 1 = guasto in corso, 0 = normale

failure_windows = [
    ("2020-04-18 00:00:00", "2020-04-18 23:59:00"),
    ("2020-05-29 23:30:00", "2020-05-30 06:00:00"),
    ("2020-06-05 10:00:00", "2020-06-07 14:30:00"),
    ("2020-07-15 14:30:00", "2020-07-15 19:00:00"),
]

# Costruiamo la condizione come OR di tutti gli intervalli
failure_condition = None
for start, end in failure_windows:
    window_cond = (F.col("timestamp") >= start) & (F.col("timestamp") <= end)
    failure_condition = window_cond if failure_condition is None else (failure_condition | window_cond)

if failure_condition is not None:
    df_raw = df_raw.withColumn("target", F.when(failure_condition, 1).otherwise(0))
    df_raw.cache()

    n_rows = df_raw.count()
    n_cols = len(df_raw.columns)
    n_failures = df_raw.filter(F.col("target") == 1).count()

    print(f"Righe      : {n_rows:,}")
    print(f"Colonne    : {n_cols}")
    print(f"Guasti (1) : {n_failures:,} ({n_failures/n_rows*100:.2f}%)")
    print(f"Normali (0): {n_rows - n_failures:,} ({(n_rows-n_failures)/n_rows*100:.2f}%)")
    print(f"Tempo caricamento: {time.time() - t0:.1f}s")

**Output**
| Metrica | Valore |
|---|---|
| Righe totali | 1,516,948 |
| Colonne | 18 (15 feature + index + timestamp + target) |
| Guasti (target=1) | 29,954 (1.97%) |
| Normali (target=0) | 1,486,994 (98.03%) |
| Tempo caricamento | 8.7s |

**Osservazioni**
Il dataset risulta **fortemente sbilanciato**: i guasti rappresentano meno del 2%
delle osservazioni, il che riflette fedelmente la realtà operativa — i guasti
sono eventi rari rispetto al normale funzionamento del compressore.

Questo sbilanciamento ha due implicazioni dirette per la nostra analisi:
- **Metrica**: useremo **F1-score** invece della accuracy. Con il 98% di normali,
  un modello che predice sempre 0 avrebbe accuracy del 98% pur essendo inutile.
- **Iniezione del rumore con Pucktrick**: il rumore sulla classe minoritaria (guasti)
  sarà potenzialmente più destabilizzante rispetto alla classe maggioritaria —
  questo è uno degli aspetti più interessanti da analizzare negli esperimenti.

**Nota metodologica — Perché usiamo F1-score?**
La F1-score è la metrica principale che useremo per valutare i nostri modelli.
Per capire perché, è necessario introdurre due concetti: **precision** e **recall**.

Un modello che rileva guasti può sbagliare in due modi distinti:
- **Falso positivo**: predice un guasto che non c'è → intervento di manutenzione inutile
- **Falso negativo**: non rileva un guasto reale → guasto ignorato, potenzialmente pericoloso

**Precision** risponde a: *di tutti i guasti predetti, quanti erano reali?*
**Recall** risponde a: *di tutti i guasti reali, quanti ne ho trovati?*

La F1-score è la media armonica delle due, e vale tra 0 e 1 (1 = perfetto):

$$F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$$

**Perché non usiamo l'accuracy?**

Sul nostro dataset un modello che predice **sempre "normale"** otterrebbe:
- Accuracy = **98%** ✅ — sembra eccellente
- F1-score ≈ **0** ❌ — non ha rilevato nemmeno un guasto

L'accuracy è fuorviante quando il dataset è sbilanciato perché premia il modello
anche se ignora completamente la classe minoritaria — in questo caso i guasti,
che sono esattamente quelli che ci interessa rilevare.
La F1-score invece crolla a zero se i guasti non vengono trovati, risultando
una metrica molto più onesta e utile per il nostro scopo.

#### Step 3 — Esplorazione iniziale del dataset

Prima di costruire qualsiasi modello è fondamentale capire com'è fatto il dataset:
che tipo di valori hanno le feature, se ci sono anomalie, e come si comportano
i sensori nel tempo. Questa fase guida tutte le scelte successive.

**Cosa analizziamo**
- **Schema**: tipo di ogni colonna (numerico, stringa, ecc.)
- **Statistiche descrittive**: media, deviazione standard, min e max di ogni sensore —
  utile per capire i range operativi normali del compressore
- **Missing values**: confermiamo che non ce ne siano (come indicato nella documentazione)
- **Distribuzione target**: visualizziamo lo sbilanciamento tra guasti e normali
- **Andamento temporale**: plottiamo i sensori nel tempo evidenziando le finestre di guasto —
  ci permette di vedere visivamente se i sensori cambiano comportamento durante i guasti
- **Matrice di correlazione**: identifica feature ridondanti o molto correlate tra loro,
  informazione utile per la successiva fase di feature selection

In [ ]:
# ── Step 3 · Esplorazione iniziale ────────────────────────────────────────

# 3.1 — Schema e tipi delle colonne
print("=== SCHEMA ===")
df_raw.printSchema()

# 3.2 — Statistiche descrittive delle feature numeriche
print("=== STATISTICHE DESCRITTIVE ===")
feature_cols = ["TP2","TP3","H1","DV_pressure","Reservoirs",
                "Oil_temperature","Motor_current","COMP","DV_eletric",
                "Towers","MPG","LPS","Pressure_switch","Oil_level","Caudal_impulses"]
df_raw.select(feature_cols).describe().show(truncate=False)

# 3.3 — Verifica missing values
print("=== MISSING VALUES ===")
for col in feature_cols:
    n_null = df_raw.filter(F.col(col).isNull()).count()
    print(f"  {col:<20}: {n_null}")

# 3.4 — Distribuzione target (grafico)
target_dist = df_raw.groupBy("target").count().orderBy("target").toPandas()

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(
    ["Normale (0)", "Guasto (1)"],
    target_dist["count"],
    color=["#2D6A9F", "#E07B39"],
    edgecolor="white", linewidth=1.5
)
for bar, cnt in zip(bars, target_dist["count"]):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.02,
            f"{cnt:,}", ha="center", va="bottom", fontsize=10)
ax.set_title("Distribuzione classi — MetroPT-3", fontsize=13, fontweight="bold")
ax.set_ylabel("Numero di campioni")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/01_target_distribution.png")
plt.show()
print(f"→ Salvato: {OUTPUT_DIR}/01_target_distribution.png")

# 3.5 — Andamento temporale dei sensori analogici
pdf_sample = df_raw.select("timestamp", "TP2", "TP3", "Motor_current",
                            "Oil_temperature", "target") \
                    .orderBy("timestamp") \
                    .limit(50000) \
                    .toPandas()
pdf_sample["timestamp"] = pd.to_datetime(pdf_sample["timestamp"])

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
sensors = ["TP2", "TP3", "Motor_current", "Oil_temperature"]
colors  = ["#2D6A9F", "#6BAE75", "#E07B39", "#9B59B6"]

for ax, sensor, color in zip(axes, sensors, colors):
    ax.plot(pdf_sample["timestamp"], pdf_sample[sensor],
            color=color, linewidth=0.5, label=sensor)
    # Evidenzia le zone di guasto in rosso
    fault_mask = pdf_sample["target"] == 1
    ax.fill_between(pdf_sample["timestamp"], pdf_sample[sensor].min(),
                    pdf_sample[sensor].max(),
                    where=fault_mask, color="red", alpha=0.3, label="Guasto")
    ax.set_ylabel(sensor, fontsize=10)
    ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Timestamp", fontsize=10)
fig.suptitle("Andamento temporale sensori analogici\n(primi 50.000 campioni)",
                fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02_time_series_sensors.png")
plt.show()
print(f"→ Salvato: {OUTPUT_DIR}/02_time_series_sensors.png")

# 3.6 — Matrice di correlazione
corr_pdf = df_raw.select(feature_cols).limit(100000).toPandas().corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_pdf, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, linewidths=0.5, annot_kws={"size": 7})
ax.set_title("Matrice di correlazione — feature sensori", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/03_correlation_matrix.png")
plt.show()
print(f"→ Salvato: {OUTPUT_DIR}/03_correlation_matrix.png")

In [ ]:
# 3.7 — Andamento temporale in una finestra con guasto (aprile 2020)
pdf_fault = df_raw.select("timestamp", "TP2", "TP3", "Motor_current",
                           "Oil_temperature", "target") \
                  .filter(
                      (F.col("timestamp") >= "2020-04-17 00:00:00") &
                      (F.col("timestamp") <= "2020-04-19 23:59:00")
                  ) \
                  .orderBy("timestamp") \
                  .toPandas()
pdf_fault["timestamp"] = pd.to_datetime(pdf_fault["timestamp"])

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
sensors = ["TP2", "TP3", "Motor_current", "Oil_temperature"]
colors  = ["#2D6A9F", "#6BAE75", "#E07B39", "#9B59B6"]

for ax, sensor, color in zip(axes, sensors, colors):
    ax.plot(pdf_fault["timestamp"], pdf_fault[sensor],
            color=color, linewidth=0.6, label=sensor)
    fault_mask = pdf_fault["target"] == 1
    ax.fill_between(pdf_fault["timestamp"],
                    pdf_fault[sensor].min(),
                    pdf_fault[sensor].max(),
                    where=fault_mask, color="red", alpha=0.3, label="Guasto")
    ax.set_ylabel(sensor, fontsize=10)
    ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Timestamp", fontsize=10)
fig.suptitle("Comportamento sensori durante il Guasto #1 — 18 Aprile 2020",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02b_fault_window.png")
plt.show()
print(f"→ Salvato: {OUTPUT_DIR}/02b_fault_window.png")

**Step 3 — Risultati esplorazione iniziale**

***Schema***
Tutte le 15 feature sono di tipo `double` (numeri decimali), il timestamp è
riconosciuto correttamente e il target è `integer`. Nessuna anomalia di tipo.

***Missing values***
**Zero missing values** su tutte le feature — il dataset è pulito e non richiede
nessuna strategia di imputazione. Questo è confermato anche dalla documentazione ufficiale.

***Statistiche descrittive***
Le feature si dividono in due gruppi con caratteristiche molto diverse:

**Sensori analogici** (TP2, TP3, H1, Motor_current, Oil_temperature, Reservoirs):
valori continui su scale diverse — la temperatura va da 15°C a 89°C, le pressioni
da 0 a 10 bar. Sarà necessario **normalizzarli** nel preprocessing, altrimenti le
feature con valori numericamente più grandi dominerebbero sulle altre durante il training.

**Sensori digitali** (COMP, DV_eletric, Towers, MPG, LPS, Pressure_switch, Oil_level,
Caudal_impulses): valori binari tra 0 e 1, già sulla stessa scala, nessuna normalizzazione necessaria.

**Distribuzione classi**
Lo sbilanciamento è estremo: **98% normali, 2% guasti**. Confermato visivamente
dal grafico — la barra dei guasti è quasi invisibile. Questo giustifica la scelta
di F1-score come metrica principale.

***Andamento temporale — comportamento normale***
I primi 50.000 campioni (prime 2 settimane di febbraio) mostrano il comportamento
normale del compressore: TP2 e TP3 oscillano ciclicamente tra valori alti e bassi,
corrispondenti ai cicli di accensione e spegnimento del compressore.
L'Oil_temperature è invece un segnale lento e continuo, tipico delle variabili termiche.
Non sono presenti guasti in questa finestra temporale — il primo guasto avviene ad aprile.

***Andamento temporale — finestra con guasto***
Il grafico della finestra del **Guasto #1 (18 Aprile 2020)** evidenzia in rosso
il periodo di air leak. È possibile osservare visivamente come i sensori cambino
comportamento durante il guasto rispetto al funzionamento normale — questa è
esattamente la firma che i nostri modelli dovranno imparare a riconoscere.

***Matrice di correlazione***
Emergono tre osservazioni importanti:

**Correlazioni altissime tra sensori analogici**: TP2, H1, COMP, DV_eletric e MPG
hanno correlazioni vicine a 1 o -1 tra loro — misurano sostanzialmente lo stesso
fenomeno fisico da angolazioni diverse. Il modello potrebbe considerarne alcune ridondanti.

**Oil_level e Caudal_impulses hanno celle vuote**: significa che la loro varianza
è quasi zero — sono costanti per quasi tutto il dataset e non portano informazione
utile al modello. Saranno candidate all'eliminazione nel preprocessing.

**DV_pressure e Pressure_switch sono correlate negativamente (-0.73)**: coerente
con il loro significato fisico — quando la pressione scende, il pressure switch si attiva.

#### Step 4